# 01 Data preparation

Processing xlsx files from data folder into suitable inputs and generate other input files

In [1]:
import pandas as pd
import numpy as np
import os
import pickle

In [2]:
cell_line ='BC3C'
data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_LINCS/00_outputs_2020_{cell_line}/"
info_dir = data_dir
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_oct_aug/00_outputs_2020_{cell_line}/"

#os.makedirs(info_dir, exist_ok = True)
os.makedirs(out_dir, exist_ok = True)

## Modules

Load data about modules and drugs.

In [3]:
os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx")

'/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_LINCS/00_outputs_2020_BC3C/ALL_DATA_2020_oct25.xlsx'

In [4]:
### DATA
### remove whitespaces in names (modules), remove duplicates

modules_df = pd.read_excel(
    os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx"), sheet_name = "modules", index_col = 0)
# display(modules_df)

selected_modules = modules_df.index.tolist()
print(len(selected_modules), ' - Size after reading')

# remove duplicates
selected_modules = modules_df.index.unique().tolist()
print(len(selected_modules), ' - Size after remove duplicates')

# remove whitespaces in modules' names 
selected_modules = [d.strip() for d in selected_modules]

print('Selected modules list: ', len(selected_modules), selected_modules)


12  - Size after reading
12  - Size after remove duplicates
Selected modules list:  12 ['CDK1_2', 'CDK4_6', 'EGFR', 'PI3K', 'FGFR', 'TOP2A', 'p53', 'Src', 'Estrogen', 'Androgen', 'TGFb', 'SMAD3']


In [5]:
### DATA
### remove whitespaces in names (modules, drugs), remove duplicates
### check the dimensions of the indicator IC50 1 uM = 1000 nM
### copy-paste as values, numbers, no formulas

IC50_df = pd.read_excel(
    os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx"), sheet_name = "IC50s")
IC50_df.drop(columns=['Unnamed: 3','Unnamed: 4'],inplace=True)

print(len(IC50_df.index), ' - Size after reading')
# display(IC50_df)

# rename
IC50_df = IC50_df.rename(columns = {"IC50, uM": "IC50"})

# manually correcting value IC50, Example  for IOX2 -> 30 nM  
#IC50_df.loc[IC50_df.index == 'IOX2', IC50_df.columns == 'IC50'] = 30/1000

# remove non-selected modules, modules' names with whitespaces or empty
IC50_df = IC50_df[IC50_df.Module.isin(selected_modules)]

print(len(IC50_df.index), ' - Size after remove modules')
# display(IC50_df)

# remove duplicates 
# Considering certain columns is optional. 
# Indexes, including time indexes are ignored.
IC50_df = IC50_df.drop_duplicates()

print(len(IC50_df.index), ' - Size after remove duplicates')
display(IC50_df)

42  - Size after reading
42  - Size after remove modules
42  - Size after remove duplicates


,Drug,Module,IC50
0,flufenamic-acid,Androgen,3.00000
1,nandrolone,Androgen,9.00000
2,oxandrolone,Androgen,190.30000
3,testosterone-enanthate,Androgen,200.00000
4,testosterone-propionate,Androgen,124.00000
5,JNJ-7706621,CDK1_2,0.02700
6,PHA-793887,CDK1_2,0.18000
7,roscovitine,CDK1_2,2.00000
8,alvocidib,CDK4_6,0.12000
9,palbociclib,CDK4_6,0.04500


In [6]:
modules = IC50_df.Module.unique().tolist()

print('IC50_df  modules list: ', len(modules), modules)
print()
print('Selected modules list: ', len(selected_modules), selected_modules)

### CHECK
print()
print('CHECK: ', len(selected_modules),'=?', len(modules))

n_modules = len(modules)


IC50_df  modules list:  12 ['Androgen', 'CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'TGFb', 'SMAD3']

Selected modules list:  12 ['CDK1_2', 'CDK4_6', 'EGFR', 'PI3K', 'FGFR', 'TOP2A', 'p53', 'Src', 'Estrogen', 'Androgen', 'TGFb', 'SMAD3']

CHECK:  12 =? 12


In [7]:
drugs = IC50_df.Drug.tolist()
print(len(drugs), ' - Size after reading')

# remove duplicates
drugs = IC50_df.Drug.unique().tolist()
print(len(drugs), ' - Size after remove duplicates')

# remove whitespaces in drugs' names (necessary for some)
drugs = [d.strip() for d in drugs]

# remove duplicates after remove whitespaces
drugs = list(set(drugs))
print(len(drugs), ' - Size after remove duplicates without whitespaces')

print('Drugs list: ', len(drugs), drugs)

n_drugs = len(drugs)

42  - Size after reading
42  - Size after remove duplicates
42  - Size after remove duplicates without whitespaces
Drugs list:  42 ['estradiol-cypionate', 'nutlin-3', 'sorafenib', 'NVP-BEZ235', 'AZD-8055', 'idarubicin', 'PI-103', 'Agent1', 'PHA-793887', 'KU-0063794', 'erlotinib', 'GDC-0349', 'Agent2', 'alvocidib', 'SAR405838', 'JNJ-7706621', 'testosterone-propionate', 'HLI-373', 'AMG-232', 'afatinib', 'flufenamic-acid', 'roscovitine', 'AS-605240', 'vandetanib', 'lapatinib', 'dasatinib', 'epirubicin', 'serdemetan', 'RITA', 'nandrolone', 'ponatinib', 'testosterone-enanthate', 'LY-294002', 'daunorubicin', 'palbociclib', 'gefitinib', 'raloxifene', 'taselisib', 'masitinib', 'dienestrol', 'mitoxantrone', 'oxandrolone']


## L1000 meta data

Get sig_id for selected drugs.

In [8]:
sig_info_df = pd.read_excel(os.path.join(data_dir, f"sig_info_2020_{cell_line}.xlsx"), index_col = 0)

display(sig_info_df)

,cell,plate,time,level_3_samples,samples_number,pert_type,pert_drug,targets,targets_number,dose,dose_float
level_5_sig_id,,,,,,,,,,,
ASG002_BC3C_24H:A03,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A03,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A04,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A04,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A05,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A05,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A06,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A06,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:J13,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:J13,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
...,...,...,...,...,...,...,...,...,...,...,...
MOAR012_BC3C_24H:P20,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P20,1,trt_cp,BAY-61-3606,NaN,0,3.33 uM,3.33
MOAR012_BC3C_24H:P21,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P21,1,trt_cp,BAY-61-3606,NaN,0,1.11 uM,1.11
MOAR012_BC3C_24H:P22,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P22,1,trt_cp,ethaverine,NaN,0,10 uM,10.00


In [9]:
# now filtering so only the required drugs are present
sig_info_df = sig_info_df.loc[sig_info_df.pert_drug.isin(drugs)]

# here's what we have now
display(sig_info_df)

,cell,plate,time,level_3_samples,samples_number,pert_type,pert_drug,targets,targets_number,dose,dose_float
level_5_sig_id,,,,,,,,,,,
ASG002_BC3C_24H:A10,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A10,1,trt_cp,taselisib,PIK3CA,1,10 uM,10.00
ASG002_BC3C_24H:A11,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A11,1,trt_cp,taselisib,PIK3CA,1,1.11 uM,1.11
ASG002_BC3C_24H:A19,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A19,1,trt_cp,AS-605240,PIK3CG,1,10 uM,10.00
ASG002_BC3C_24H:A20,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A20,1,trt_cp,AS-605240,PIK3CG,1,1.11 uM,1.11
ASG002_BC3C_24H:A21,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A21,1,trt_cp,AS-605240,PIK3CG,1,0.12 uM,0.12
...,...,...,...,...,...,...,...,...,...,...,...
MOAR011_BC3C_24H:C11,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:C11,1,trt_cp,testosterone-enanthate,AR,1,3.33 uM,3.33
MOAR011_BC3C_24H:F07,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:F07,1,trt_cp,serdemetan,MDM2,1,10 uM,10.00
MOAR011_BC3C_24H:F08,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:F08,1,trt_cp,serdemetan,MDM2,1,3.33 uM,3.33


In [10]:
#previous removed
#rows_remove = ['ASG002_BC3C_24H:F04', 'ASG002_BC3C_24H:F05','ASG002_BC3C_24H:L02',
#               'ASG002_BC3C_24H:O24','ASG002_B                         C3C_24H:L07','ASG002_BC3C_24H:L09',
#               'MOAR010_BC3C_24H:D02', 'ASG002_BC3C_24H:F01','MOAR010_BC3C_24H:D01',
#               'ASG002_BC3C_24H:N19','ASG002_BC3C_24H:N21','ASG002_BC3C_24H:N24',
#               'ASG002_BC3C_24H:I19','ASG002_BC3C_24H:I21','ASG002_BC3C_24H:L17',
#               'ASG002_BC3C_24H:M23','MOAR008_BC3C_24H:L03','MOAR010_BC3C_24H:L20','MOAR011_BC3C_24H:J10',
#               'MOAR008_BC3C_24H:L08','MOAR009_BC3C_24H:C10','MOAR010_BC3C_24H:A13','MOAR010_BC3C_24H:A14','MOAR011_BC3C_24H:F09',
#               'ASG002_BC3C_24H:G01','ASG002_BC3C_24H:P20']

In [11]:
rows_remove = ['ASG002_BC3C_24H:G01',
 'ASG002_BC3C_24H:F15',
 'ASG002_BC3C_24H:N19',
 'ASG002_BC3C_24H:N22',
 'ASG002_BC3C_24H:F15',
 #'ASG002_BC3C_24H:I08',
 'ASG002_BC3C_24H:I19',
 'ASG002_BC3C_24H:B12',
 'MOAR008_BC3C_24H:L07',
 'MOAR008_BC3C_24H:L09',
 'ASG002_BC3C_24H:F04',
 'ASG002_BC3C_24H:P17',
 'ASG002_BC3C_24H:O24',
 'MOAR010_BC3C_24H:D02',
 'ASG002_BC3C_24H:L18',
 'MOAR008_BC3C_24H:L03',
 'MOAR010_BC3C_24H:L21']

Manually remove  few inhibition from data set, since it does differ from the other data points.

inhib_to_filter = "PF-03758309"
dose_to_filter = 10

id_to_filter = sig_info_df[
    np.logical_and(
        sig_info_df.pert_drug == inhib_to_filter,
        sig_info_df.dose_float == dose_to_filter,
    )
].index.values

print(id_to_filter)

sig_info_df = sig_info_df[~sig_info_df.index.isin(id_to_filter)]
display(sig_info_df)

inhib_to_filter = "roscovitine"
dose_to_filter = 3.33

id_to_filter = sig_info_df[
    np.logical_and(
        sig_info_df.pert_drug == inhib_to_filter,
        sig_info_df.dose_float == dose_to_filter,
    )
].index.values

print(id_to_filter)

sig_info_df = sig_info_df[~sig_info_df.index.isin(id_to_filter)]
display(sig_info_df)

In [12]:
sig_info_df.drop(rows_remove,inplace=True)

In [13]:
exp_ids = sig_info_df.index.unique().tolist()
print('Experiments ids list: ', len(exp_ids))

n_experiments = len(exp_ids)

Experiments ids list:  104


In [14]:
n_experiments = len(exp_ids)
print('Experiments ids list: ', len(exp_ids))


Experiments ids list:  104


Confirm data by checking the drugs of interest against the filtered L1000 meta data.

In [15]:
print(f"Number of drugs of interest:\t{len(drugs)}")
#print(f'Number of drugs in L1000 data:\t{len(sig_info_df.value_counts("drugs"))}')

#sig_info_df.value_counts("drugs")

Number of drugs of interest:	42


## L1000 data

In [16]:
Data_norm_df = pd.read_excel(os.path.join(data_dir, f"Data_norm_2020_{cell_line}.xlsx"), index_col = 0)
display(Data_norm_df)

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
ASG002_BC3C_24H:A03,-0.191254,-0.055246,0.039596,-0.256266,-0.040419,-0.590523,-0.159396,-0.074319,0.457981,0.409608,...,0.543203,0.494266,-0.011923,-0.225931,0.285054,-0.775246,0.166031,-0.024873,0.238723,0.284204
ASG002_BC3C_24H:A04,-0.265754,-0.317496,0.118696,-0.136665,-0.301569,-0.403023,0.124804,-0.036470,0.311931,0.660457,...,-0.565096,-0.088634,0.122977,-0.047931,0.141804,0.129054,-0.028819,-0.028773,-0.253627,-0.752646
ASG002_BC3C_24H:A05,-0.181954,-0.081597,-0.210304,1.559535,-0.019019,-0.457423,0.071404,0.074080,-0.356119,0.498808,...,0.226104,-0.228034,-0.121023,-0.075331,-0.133146,0.355054,0.022831,-0.084073,0.283123,-0.894896
ASG002_BC3C_24H:A06,0.033446,0.042404,-0.150154,-0.093165,0.053180,-0.053823,0.087704,0.167681,-0.601569,0.383308,...,-0.608596,-0.228835,0.072777,0.082970,-0.570996,2.847754,-0.211670,-0.067273,0.081723,0.338704
ASG002_BC3C_24H:J13,0.204446,0.180704,0.089096,-0.054666,0.053381,0.044877,-0.277396,-0.157419,0.535681,-3.933493,...,-0.318397,0.122265,-0.134323,-0.088931,-0.067996,-0.515847,-0.005069,0.067527,0.002223,0.204904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MOAR012_BC3C_24H:P20,0.647151,0.211700,-0.979200,0.597350,-0.375751,0.388300,0.394524,0.120151,-0.166775,-1.129125,...,-1.598475,-0.552750,0.515151,0.120800,0.082675,0.529700,0.383225,-0.207225,2.268450,-1.248500
MOAR012_BC3C_24H:P21,0.171800,0.046300,-0.145550,-0.295150,0.030849,0.420951,0.222075,0.179800,0.274724,-0.423975,...,-1.650575,0.203600,-0.003250,-0.064800,-0.037675,0.076499,0.201825,0.416875,0.287450,-0.971700
MOAR012_BC3C_24H:P22,0.648700,0.058749,-0.031700,0.408249,-0.753950,0.332200,-0.357525,-0.107650,-0.213575,0.074225,...,0.127625,0.031600,0.103250,-0.249600,0.046375,1.486200,0.440325,0.090075,-0.031650,-0.944300
MOAR012_BC3C_24H:P23,0.090499,-0.469300,-0.611800,0.873550,-0.788450,-0.097199,-0.366575,-0.490600,-0.624675,-0.009275,...,0.054676,-0.596050,0.084600,0.444700,0.431375,-0.921501,0.044926,0.716076,-0.000900,-1.106700


In [17]:
Data_norm_df = Data_norm_df[Data_norm_df.index.isin(exp_ids)]

# arrange experiments in same order as in list
Data_norm_df["sort_col"] = Data_norm_df.index.map({val: i for i, val in enumerate(exp_ids)})
Data_norm_df = Data_norm_df.sort_values("sort_col")
Data_norm_df = Data_norm_df.drop("sort_col", axis = 1)

# transpose
Data_norm_df = Data_norm_df.T

display(Data_norm_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:L20,MOAR011_BC3C_24H:C01,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10
AARS,-0.496854,0.288446,0.189747,-0.016454,0.080746,0.282346,0.326246,0.303046,0.387546,0.434746,...,-0.012317,0.215100,-0.178100,-0.007000,0.024000,0.007400,0.584300,0.114500,-0.268751,0.031399
ABCB6,-0.658596,-0.142196,-0.075397,-0.383796,-0.199996,-0.074197,0.108804,-0.399196,-0.227496,-0.041146,...,-0.661323,0.127850,0.081150,-0.026850,-0.152851,0.122550,0.006950,-0.037150,-0.014150,-0.029350
ABCC5,-0.080204,0.231996,-0.329354,-0.225204,0.278446,0.034696,0.396596,-0.255704,-0.254154,-0.002004,...,-0.051357,0.197500,0.119350,0.212450,-0.177250,0.138650,-0.238750,0.409550,0.221350,0.568950
ABCF1,0.202535,0.602335,0.403335,0.313134,-0.083265,-0.056365,-0.387216,0.097235,0.540634,-0.262666,...,-0.555990,-0.036825,-0.080325,0.059175,0.307175,0.580075,0.142675,0.077875,0.695325,0.322875
ABCF3,-0.520919,-0.192819,0.001032,-0.096419,0.210881,-0.731118,0.095381,0.313481,-0.078018,-0.257669,...,0.447982,-0.378251,0.003950,0.127300,-0.197401,-0.128351,-0.468050,-0.140650,-0.353450,-0.012850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF395,1.796254,1.773154,0.668354,0.825354,0.350654,0.205954,-0.998346,0.235354,0.230653,0.440854,...,-0.050680,0.300250,-0.265350,-0.380850,-0.330950,-1.844650,0.257250,-1.828550,-1.550550,-0.612350
ZNF451,-0.244519,0.116732,0.058081,-0.178169,-0.080619,-0.139119,0.064681,-0.083019,-0.272119,-0.164119,...,-0.083718,-0.324500,0.080550,-0.114950,-0.337650,-0.024950,0.080650,0.119950,0.044250,-0.151550
ZNF586,0.097627,0.061027,-0.337573,0.098427,-0.338173,0.017627,-0.197423,-0.078473,-0.300473,-0.147773,...,-0.123664,-0.304650,-0.259400,0.163600,-0.428700,-0.069200,-0.561900,0.202100,0.043000,0.073700
ZNF589,0.608573,0.106123,-0.014126,-0.003677,-0.123477,-0.243377,0.044623,-0.021477,0.201823,-0.413227,...,0.007359,-0.315500,-0.013900,-0.035050,1.836400,-0.223300,-0.557600,-0.364600,-0.531500,-0.343400


### Concatenating TGFbRin and SMAD3in LFC2

In [18]:
files_path ='/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bc3c_oct'

In [19]:
counts = pd.read_csv(
    '/home/jing/Phd_project/project_UCD_blca/blca_DATA/blca_DATA_bc3c_oct/gene_count.xls',
    sep='\t',   
    header=0,index_col=0
)
display(counts)

,D01,D02,D03,T01,T02,T03,S01,S02,S03,gene_name,gene_chr,gene_start,gene_end,gene_strand,gene_length,gene_biotype,gene_description,tf_family
gene_id,,,,,,,,,,,,,,,,,,
ENSG00000156508,236599,217166,230137,246465,277179,265535,291410,287477,279305,EEF1A1,6,73515750,73523797,-,5948,protein_coding,eukaryotic translation elongation factor 1 alp...,-
ENSG00000186081,173372,151121,162044,168813,186988,177195,144747,149534,158903,KRT5,12,52514575,52520687,-,4292,protein_coding,keratin 5 [Source:HGNC Symbol;Acc:HGNC:6442],-
ENSG00000080824,147634,114075,138470,120147,137207,134926,93481,97859,87311,HSP90AA1,14,102080738,102139699,-,5248,protein_coding,heat shock protein 90 alpha family class A mem...,-
ENSG00000187134,105703,95969,88389,103744,120824,114376,95104,108179,106387,AKR1C1,10,4963253,4983283,+,8765,protein_coding,aldo-keto reductase family 1 member C1 [Source...,-
ENSG00000196139,93872,97803,90777,109018,125059,116421,101868,96017,98182,AKR1C3,10,5035354,5107686,+,4532,protein_coding,aldo-keto reductase family 1 member C3 [Source...,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000275063,0,0,0,0,0,0,0,0,0,AC233755.1,KI270726.1,41444,41876,+,351,protein_coding,immunoglobulin heavy variable 4-38-2-like [Sou...,-
ENSG00000275405,0,0,0,0,0,0,0,0,0,RF00003,KI270713.1,21861,22024,-,164,snRNA,NaN,-
ENSG00000275987,0,0,0,0,0,0,0,0,0,RF00003,KI270713.1,30437,30580,-,144,snRNA,NaN,-


In [20]:
tgfb_df = pd.read_csv(os.path.join(files_path,'augmented_Group_T_vs_D_DEGs.csv'),index_col=0)
display(tgfb_df)
smad_df  =pd.read_csv(os.path.join(files_path,'augmented_Group_S_vs_D_DEGs.csv'),index_col=0)
display(smad_df)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
ENSG00000156508,0.201061,0.173573,0.202641,0.176735,0.132901,0.257407,0.111420,0.173126,0.141177,0.082286,0.095872,0.161413,0.241042,0.222366,0.236890,0.192645,0.206299,0.107757,0.130863,0.219968
ENSG00000187134,0.021823,0.300893,0.267662,0.293666,0.201794,0.173898,0.199301,0.058201,0.273452,0.273660,0.123448,0.276374,0.303793,0.257057,0.287959,0.141542,0.255065,0.115740,0.153871,0.211205
ENSG00000196139,0.123042,0.390659,0.411170,0.292168,0.266489,0.189001,0.231022,0.159906,0.412510,0.323906,0.131149,0.397332,0.300021,0.334547,0.263392,0.169997,0.279720,0.150318,0.257796,0.364778
ENSG00000075624,-0.292043,-0.200682,-0.197221,-0.272745,-0.242856,-0.416965,-0.231833,-0.266039,-0.165688,-0.156133,-0.213593,-0.198982,-0.339592,-0.307785,-0.384031,-0.275137,-0.345786,-0.220557,-0.164550,-0.228457
ENSG00000151632,0.096060,0.335307,0.324412,0.254603,0.220380,0.124327,0.173548,0.084769,0.330582,0.275151,0.092567,0.332995,0.250335,0.265957,0.193553,0.124103,0.240624,0.113041,0.189223,0.282659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000143590,1.357461,0.838161,-0.009501,1.151821,0.968775,1.256215,1.248668,1.150430,0.523499,1.075849,0.871366,0.646154,1.186204,2.253030,1.775349,1.269010,1.445227,0.825412,0.077452,0.868504
ENSG00000224186,0.901672,1.283645,1.193499,1.233316,1.230683,1.222674,1.226041,0.627548,1.705221,1.082202,0.116156,1.573791,1.197308,1.650068,0.848380,0.640848,1.695109,0.577007,0.938254,1.156662
ENSG00000267698,0.375276,1.096940,1.222448,1.185693,0.643781,1.137254,1.154811,0.726687,1.413238,1.072395,0.319942,1.108474,1.343758,1.760660,1.252416,0.968138,1.073060,0.480426,1.024734,1.200079
ENSG00000105427,1.479687,0.996351,1.235749,1.190506,0.612649,1.825561,0.772510,1.249845,0.795893,0.564786,0.616805,0.854932,1.707409,1.904058,1.630702,1.535507,1.285501,0.726487,0.909379,1.534727


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
ENSG00000156508,0.267818,0.250431,0.258339,0.181589,0.250753,0.219754,0.259317,0.200100,0.236601,0.227299,0.330536,0.244935,0.227655,0.239768,0.164381,0.246356,0.214813,0.262664,0.294040,0.320645
ENSG00000186081,-0.270659,-0.214579,-0.177583,-0.122605,-0.238233,-0.140340,-0.112750,-0.102218,-0.114194,-0.177293,-0.259021,-0.224725,-0.077319,-0.205527,-0.099922,-0.103619,-0.103381,-0.179374,-0.202944,-0.251690
ENSG00000080824,-0.648882,-0.609840,-0.621077,-0.519491,-0.620831,-0.575570,-0.539669,-0.512242,-0.535387,-0.558118,-0.690562,-0.594507,-0.524116,-0.536778,-0.495778,-0.536145,-0.520466,-0.597496,-0.589405,-0.680019
ENSG00000075624,-0.553327,-0.447304,-0.530273,-0.409481,-0.429479,-0.424402,-0.449047,-0.321024,-0.462650,-0.421495,-0.581068,-0.452151,-0.449315,-0.367896,-0.355937,-0.351615,-0.408055,-0.472188,-0.447125,-0.475379
ENSG00000096384,-0.631544,-0.577096,-0.566570,-0.477981,-0.583133,-0.510339,-0.492897,-0.451967,-0.488453,-0.526770,-0.657332,-0.566211,-0.466472,-0.519741,-0.444723,-0.471725,-0.471840,-0.551671,-0.561901,-0.627111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000053702,-2.002080,-1.689436,-1.561071,0.541366,-2.684528,-2.770834,-1.487364,-2.053640,-1.062125,-1.418561,-2.635171,-3.347576,-1.455604,-2.976164,-0.683174,-2.254330,0.179689,-3.332501,-2.159110,-5.245321
ENSG00000184860,-4.867906,-3.830393,-3.064666,-2.068944,-3.196792,-2.044355,-2.590525,-1.043911,-3.145863,-3.208080,-4.767025,-3.626019,-2.830786,-3.678875,-1.930937,-1.055293,-1.382268,-3.876558,-3.180868,-3.062467
ENSG00000279259,2.519366,3.534411,2.960847,1.804856,1.837106,1.713883,2.482235,2.595317,2.774206,2.618790,5.358544,1.328676,3.249234,0.935202,1.353173,1.983776,2.747092,3.341848,2.344522,3.813338
ENSG00000227359,-4.315378,-2.366259,-2.934172,-1.755826,-2.419720,-1.921446,-2.700961,-0.303094,-2.901534,-2.372890,-3.551570,-3.395520,-2.468915,-3.105285,-1.447017,-0.934478,-1.227321,-3.039423,-2.944456,-2.242576


In [21]:
tgfb_df['symbol']= counts.loc[tgfb_df.index,'gene_name']
display(tgfb_df)
smad_df['symbol']= counts.loc[smad_df.index,'gene_name']
display(smad_df)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V12,V13,V14,V15,V16,V17,V18,V19,V20,symbol
ENSG00000156508,0.201061,0.173573,0.202641,0.176735,0.132901,0.257407,0.111420,0.173126,0.141177,0.082286,...,0.161413,0.241042,0.222366,0.236890,0.192645,0.206299,0.107757,0.130863,0.219968,EEF1A1
ENSG00000187134,0.021823,0.300893,0.267662,0.293666,0.201794,0.173898,0.199301,0.058201,0.273452,0.273660,...,0.276374,0.303793,0.257057,0.287959,0.141542,0.255065,0.115740,0.153871,0.211205,AKR1C1
ENSG00000196139,0.123042,0.390659,0.411170,0.292168,0.266489,0.189001,0.231022,0.159906,0.412510,0.323906,...,0.397332,0.300021,0.334547,0.263392,0.169997,0.279720,0.150318,0.257796,0.364778,AKR1C3
ENSG00000075624,-0.292043,-0.200682,-0.197221,-0.272745,-0.242856,-0.416965,-0.231833,-0.266039,-0.165688,-0.156133,...,-0.198982,-0.339592,-0.307785,-0.384031,-0.275137,-0.345786,-0.220557,-0.164550,-0.228457,ACTB
ENSG00000151632,0.096060,0.335307,0.324412,0.254603,0.220380,0.124327,0.173548,0.084769,0.330582,0.275151,...,0.332995,0.250335,0.265957,0.193553,0.124103,0.240624,0.113041,0.189223,0.282659,AKR1C2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000143590,1.357461,0.838161,-0.009501,1.151821,0.968775,1.256215,1.248668,1.150430,0.523499,1.075849,...,0.646154,1.186204,2.253030,1.775349,1.269010,1.445227,0.825412,0.077452,0.868504,EFNA3
ENSG00000224186,0.901672,1.283645,1.193499,1.233316,1.230683,1.222674,1.226041,0.627548,1.705221,1.082202,...,1.573791,1.197308,1.650068,0.848380,0.640848,1.695109,0.577007,0.938254,1.156662,C5orf66
ENSG00000267698,0.375276,1.096940,1.222448,1.185693,0.643781,1.137254,1.154811,0.726687,1.413238,1.072395,...,1.108474,1.343758,1.760660,1.252416,0.968138,1.073060,0.480426,1.024734,1.200079,AC002116.2
ENSG00000105427,1.479687,0.996351,1.235749,1.190506,0.612649,1.825561,0.772510,1.249845,0.795893,0.564786,...,0.854932,1.707409,1.904058,1.630702,1.535507,1.285501,0.726487,0.909379,1.534727,CNFN


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V12,V13,V14,V15,V16,V17,V18,V19,V20,symbol
ENSG00000156508,0.267818,0.250431,0.258339,0.181589,0.250753,0.219754,0.259317,0.200100,0.236601,0.227299,...,0.244935,0.227655,0.239768,0.164381,0.246356,0.214813,0.262664,0.294040,0.320645,EEF1A1
ENSG00000186081,-0.270659,-0.214579,-0.177583,-0.122605,-0.238233,-0.140340,-0.112750,-0.102218,-0.114194,-0.177293,...,-0.224725,-0.077319,-0.205527,-0.099922,-0.103619,-0.103381,-0.179374,-0.202944,-0.251690,KRT5
ENSG00000080824,-0.648882,-0.609840,-0.621077,-0.519491,-0.620831,-0.575570,-0.539669,-0.512242,-0.535387,-0.558118,...,-0.594507,-0.524116,-0.536778,-0.495778,-0.536145,-0.520466,-0.597496,-0.589405,-0.680019,HSP90AA1
ENSG00000075624,-0.553327,-0.447304,-0.530273,-0.409481,-0.429479,-0.424402,-0.449047,-0.321024,-0.462650,-0.421495,...,-0.452151,-0.449315,-0.367896,-0.355937,-0.351615,-0.408055,-0.472188,-0.447125,-0.475379,ACTB
ENSG00000096384,-0.631544,-0.577096,-0.566570,-0.477981,-0.583133,-0.510339,-0.492897,-0.451967,-0.488453,-0.526770,...,-0.566211,-0.466472,-0.519741,-0.444723,-0.471725,-0.471840,-0.551671,-0.561901,-0.627111,HSP90AB1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000053702,-2.002080,-1.689436,-1.561071,0.541366,-2.684528,-2.770834,-1.487364,-2.053640,-1.062125,-1.418561,...,-3.347576,-1.455604,-2.976164,-0.683174,-2.254330,0.179689,-3.332501,-2.159110,-5.245321,NRIP2
ENSG00000184860,-4.867906,-3.830393,-3.064666,-2.068944,-3.196792,-2.044355,-2.590525,-1.043911,-3.145863,-3.208080,...,-3.626019,-2.830786,-3.678875,-1.930937,-1.055293,-1.382268,-3.876558,-3.180868,-3.062467,SDR42E1
ENSG00000279259,2.519366,3.534411,2.960847,1.804856,1.837106,1.713883,2.482235,2.595317,2.774206,2.618790,...,1.328676,3.249234,0.935202,1.353173,1.983776,2.747092,3.341848,2.344522,3.813338,AC087741.3
ENSG00000227359,-4.315378,-2.366259,-2.934172,-1.755826,-2.419720,-1.921446,-2.700961,-0.303094,-2.901534,-2.372890,...,-3.395520,-2.468915,-3.105285,-1.447017,-0.934478,-1.227321,-3.039423,-2.944456,-2.242576,AC017074.1


In [22]:
tgfb_lfc = tgfb_df.set_index('symbol')
tgfb_lfc = tgfb_lfc.loc[tgfb_lfc.index.intersection(Data_norm_df.index)]
display(tgfb_lfc)
smad_lfc = smad_df.set_index('symbol')
smad_lfc = smad_lfc.loc[smad_lfc.index.intersection(Data_norm_df.index)]
display(smad_lfc)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
HSPA8,-0.530289,-0.477385,-0.436713,-0.528967,-0.493244,-0.606866,-0.485275,-0.472977,-0.449245,-0.440122,-0.418003,-0.477159,-0.571660,-0.600743,-0.589029,-0.499383,-0.602073,-0.443212,-0.396937,-0.483488
SPP1,0.159662,0.146542,0.142407,0.206866,0.105709,0.241887,0.111868,0.092238,0.098278,0.109076,0.085087,0.125302,0.261620,0.233034,0.224159,0.177952,0.220070,0.103411,0.095625,0.151448
TXNRD1,0.227017,0.405422,0.379856,0.314378,0.373750,0.212434,0.329854,0.237377,0.458494,0.408085,0.224692,0.443613,0.271237,0.366149,0.254673,0.193701,0.348133,0.255639,0.297679,0.361113
RPS6,0.065750,0.418455,0.452260,0.357832,0.252752,0.216795,0.212395,0.086446,0.400799,0.339476,0.136190,0.403186,0.386860,0.310118,0.304955,0.177218,0.306467,0.147238,0.265408,0.348292
SQSTM1,0.311035,0.193522,0.174994,0.219349,0.178965,0.306141,0.183410,0.269540,0.142242,0.112636,0.181588,0.163727,0.269911,0.280338,0.293796,0.293321,0.245053,0.182702,0.164228,0.241946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FGFR2,-0.656417,-0.923720,-0.718321,-0.813559,-0.905516,-0.722763,-0.784525,-0.582475,-0.941571,-0.876477,-0.463210,-0.982695,-0.786935,-1.103338,-0.863460,-0.515432,-1.031858,-0.543613,-0.466746,-0.827796
COL1A1,-0.398945,-0.894996,-0.977012,-0.820676,-0.836227,-0.657243,-0.622016,-0.214788,-1.020809,-0.914895,-0.343304,-1.061736,-0.806189,-0.771189,-0.523551,-0.255636,-0.996917,-0.523229,-0.642004,-0.727426
SERPINE1,-0.849420,-1.279898,-1.096244,-1.293849,-1.232850,-1.176774,-1.047198,-0.644365,-1.239111,-1.191251,-0.684917,-1.344047,-1.321566,-1.370653,-1.243274,-0.757512,-1.535370,-0.812538,-0.738066,-1.050789
CHAC1,2.341970,2.874138,2.668303,2.638395,2.607622,2.354635,2.525517,2.198984,2.967931,2.667660,1.906206,2.938006,2.601913,3.067016,2.484619,2.237992,2.823817,2.099446,2.290345,2.765239


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
GAPDH,-0.418152,-0.378634,-0.364679,-0.263892,-0.347446,-0.263073,-0.244382,-0.208759,-0.301027,-0.311857,-0.466722,-0.302955,-0.265909,-0.271792,-0.233447,-0.191365,-0.237988,-0.347047,-0.293119,-0.359380
HSPD1,-0.690897,-0.665134,-0.652550,-0.567147,-0.659815,-0.609033,-0.578545,-0.549631,-0.578327,-0.605370,-0.730267,-0.629661,-0.571679,-0.580900,-0.545108,-0.570301,-0.557540,-0.639824,-0.625826,-0.701300
HSPA8,-0.596020,-0.483657,-0.553548,-0.509347,-0.450671,-0.483620,-0.561689,-0.403153,-0.524346,-0.493066,-0.583500,-0.530125,-0.539842,-0.444983,-0.441469,-0.447797,-0.517435,-0.506071,-0.536500,-0.483699
SPP1,0.183543,0.160707,0.167465,0.152414,0.141925,0.156691,0.175856,0.061337,0.119418,0.116680,0.170852,0.137803,0.149321,0.075414,0.111390,0.155068,0.129231,0.120261,0.168496,0.121010
TXNRD1,-0.369827,-0.405291,-0.334293,-0.257711,-0.399601,-0.312548,-0.209274,-0.234407,-0.237951,-0.291079,-0.404563,-0.292291,-0.228591,-0.259964,-0.261069,-0.255030,-0.199669,-0.325166,-0.274368,-0.379084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CDC42,-0.954408,-1.133006,-1.025483,-0.750016,-1.203452,-0.308864,-0.369969,-0.566596,-0.732181,-0.832729,-1.402006,-0.358399,-0.148742,-0.727734,-0.468275,-0.550377,-0.827254,-0.567495,-0.956274,-0.945379
APOE,1.757115,0.958865,1.180892,0.361918,1.470654,1.152320,0.944884,0.782349,0.704361,1.015152,1.705620,1.908782,0.546101,1.553845,0.355148,0.860957,0.546481,1.397371,1.521237,2.344394
SATB1,1.838739,1.658157,1.726674,0.842176,1.095462,0.900912,0.918231,0.684494,1.300680,1.206626,2.673180,0.971805,1.317559,0.475854,0.570397,0.362617,0.930993,1.660736,0.936688,1.743425
SPDEF,-2.192339,-2.477598,-1.448971,-0.849881,-1.784082,-1.055109,-1.117169,-1.309503,-1.334458,-1.823078,-3.150591,-1.644710,-1.430505,-1.641612,-0.846442,-0.860945,-0.920115,-2.251125,-1.632585,-2.591862


In [23]:
tgfb_lfc.columns = ['TGFbRin_' + col for col in tgfb_lfc.columns]
smad_lfc.columns = ['SMAD3in_' + col for col in smad_lfc.columns]
display(tgfb_lfc)
display(smad_lfc)

,TGFbRin_V1,TGFbRin_V2,TGFbRin_V3,TGFbRin_V4,TGFbRin_V5,TGFbRin_V6,TGFbRin_V7,TGFbRin_V8,TGFbRin_V9,TGFbRin_V10,TGFbRin_V11,TGFbRin_V12,TGFbRin_V13,TGFbRin_V14,TGFbRin_V15,TGFbRin_V16,TGFbRin_V17,TGFbRin_V18,TGFbRin_V19,TGFbRin_V20
HSPA8,-0.530289,-0.477385,-0.436713,-0.528967,-0.493244,-0.606866,-0.485275,-0.472977,-0.449245,-0.440122,-0.418003,-0.477159,-0.571660,-0.600743,-0.589029,-0.499383,-0.602073,-0.443212,-0.396937,-0.483488
SPP1,0.159662,0.146542,0.142407,0.206866,0.105709,0.241887,0.111868,0.092238,0.098278,0.109076,0.085087,0.125302,0.261620,0.233034,0.224159,0.177952,0.220070,0.103411,0.095625,0.151448
TXNRD1,0.227017,0.405422,0.379856,0.314378,0.373750,0.212434,0.329854,0.237377,0.458494,0.408085,0.224692,0.443613,0.271237,0.366149,0.254673,0.193701,0.348133,0.255639,0.297679,0.361113
RPS6,0.065750,0.418455,0.452260,0.357832,0.252752,0.216795,0.212395,0.086446,0.400799,0.339476,0.136190,0.403186,0.386860,0.310118,0.304955,0.177218,0.306467,0.147238,0.265408,0.348292
SQSTM1,0.311035,0.193522,0.174994,0.219349,0.178965,0.306141,0.183410,0.269540,0.142242,0.112636,0.181588,0.163727,0.269911,0.280338,0.293796,0.293321,0.245053,0.182702,0.164228,0.241946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FGFR2,-0.656417,-0.923720,-0.718321,-0.813559,-0.905516,-0.722763,-0.784525,-0.582475,-0.941571,-0.876477,-0.463210,-0.982695,-0.786935,-1.103338,-0.863460,-0.515432,-1.031858,-0.543613,-0.466746,-0.827796
COL1A1,-0.398945,-0.894996,-0.977012,-0.820676,-0.836227,-0.657243,-0.622016,-0.214788,-1.020809,-0.914895,-0.343304,-1.061736,-0.806189,-0.771189,-0.523551,-0.255636,-0.996917,-0.523229,-0.642004,-0.727426
SERPINE1,-0.849420,-1.279898,-1.096244,-1.293849,-1.232850,-1.176774,-1.047198,-0.644365,-1.239111,-1.191251,-0.684917,-1.344047,-1.321566,-1.370653,-1.243274,-0.757512,-1.535370,-0.812538,-0.738066,-1.050789
CHAC1,2.341970,2.874138,2.668303,2.638395,2.607622,2.354635,2.525517,2.198984,2.967931,2.667660,1.906206,2.938006,2.601913,3.067016,2.484619,2.237992,2.823817,2.099446,2.290345,2.765239


,SMAD3in_V1,SMAD3in_V2,SMAD3in_V3,SMAD3in_V4,SMAD3in_V5,SMAD3in_V6,SMAD3in_V7,SMAD3in_V8,SMAD3in_V9,SMAD3in_V10,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
GAPDH,-0.418152,-0.378634,-0.364679,-0.263892,-0.347446,-0.263073,-0.244382,-0.208759,-0.301027,-0.311857,-0.466722,-0.302955,-0.265909,-0.271792,-0.233447,-0.191365,-0.237988,-0.347047,-0.293119,-0.359380
HSPD1,-0.690897,-0.665134,-0.652550,-0.567147,-0.659815,-0.609033,-0.578545,-0.549631,-0.578327,-0.605370,-0.730267,-0.629661,-0.571679,-0.580900,-0.545108,-0.570301,-0.557540,-0.639824,-0.625826,-0.701300
HSPA8,-0.596020,-0.483657,-0.553548,-0.509347,-0.450671,-0.483620,-0.561689,-0.403153,-0.524346,-0.493066,-0.583500,-0.530125,-0.539842,-0.444983,-0.441469,-0.447797,-0.517435,-0.506071,-0.536500,-0.483699
SPP1,0.183543,0.160707,0.167465,0.152414,0.141925,0.156691,0.175856,0.061337,0.119418,0.116680,0.170852,0.137803,0.149321,0.075414,0.111390,0.155068,0.129231,0.120261,0.168496,0.121010
TXNRD1,-0.369827,-0.405291,-0.334293,-0.257711,-0.399601,-0.312548,-0.209274,-0.234407,-0.237951,-0.291079,-0.404563,-0.292291,-0.228591,-0.259964,-0.261069,-0.255030,-0.199669,-0.325166,-0.274368,-0.379084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CDC42,-0.954408,-1.133006,-1.025483,-0.750016,-1.203452,-0.308864,-0.369969,-0.566596,-0.732181,-0.832729,-1.402006,-0.358399,-0.148742,-0.727734,-0.468275,-0.550377,-0.827254,-0.567495,-0.956274,-0.945379
APOE,1.757115,0.958865,1.180892,0.361918,1.470654,1.152320,0.944884,0.782349,0.704361,1.015152,1.705620,1.908782,0.546101,1.553845,0.355148,0.860957,0.546481,1.397371,1.521237,2.344394
SATB1,1.838739,1.658157,1.726674,0.842176,1.095462,0.900912,0.918231,0.684494,1.300680,1.206626,2.673180,0.971805,1.317559,0.475854,0.570397,0.362617,0.930993,1.660736,0.936688,1.743425
SPDEF,-2.192339,-2.477598,-1.448971,-0.849881,-1.784082,-1.055109,-1.117169,-1.309503,-1.334458,-1.823078,-3.150591,-1.644710,-1.430505,-1.641612,-0.846442,-0.860945,-0.920115,-2.251125,-1.632585,-2.591862


In [24]:
Data_norm_df[tgfb_lfc.columns] = 0
Data_norm_df[smad_lfc.columns] = 0

In [25]:
common_idx = Data_norm_df.index.intersection(tgfb_lfc.index)
Data_norm_df.loc[common_idx, tgfb_lfc.columns] = tgfb_lfc.loc[common_idx, tgfb_lfc.columns]


/tmp/ipykernel_3987877/2713876191.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 5.28251709e-01  2.06263668e-01  1.01446107e-02  3.36793362e-01
  3.45504087e-01 -2.44635057e-01 -8.50516973e-02 -3.17276999e-01
 -7.17139948e-01  1.61261127e-01 -1.96195405e-01 -1.91874850e-01
 -4.49587839e-01  3.78476943e-01 -9.20940423e-02  3.06889987e-01
 -1.59735062e-01 -2.58191709e-01  3.04920285e-01 -3.18172339e-01
 -5.89846350e-02 -1.98023268e-01 -2.62904916e-01 -2.02040768e-01
  1.55948464e-01 -2.14846524e-01  2.95102511e-01 -5.29210430e-01
 -3.44309412e-01 -1.76411990e-01 -7.37473402e-01 -1.77229244e-01
 -1.05339946e-01 -1.56901404e-01  1.62048713e-01 -1.55048147e-02
 -1.73165075e-01  1.57906385e-01  1.20529824e-01  2.00503064e-01
 -2.74975987e-01 -4.22271150e-01 -3.59730476e-02 -3.61780682e-01
  3.20912593e-01 -2.29496093e-01 -9.65260750e-02 -2.31077445e-01
 -2.38106204e-01  4.96045035e-01 -2.60437102e-01  2

In [26]:
common_idx = Data_norm_df.index.intersection(smad_lfc.index)
Data_norm_df.loc[common_idx, smad_lfc.columns] = smad_lfc.loc[common_idx, smad_lfc.columns]

/tmp/ipykernel_3987877/1508837703.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.80006032 -0.38077041  0.6067599   0.14613308  1.04119834  0.64850065
  0.33453216 -0.3657465  -0.14819256 -0.93501364  0.2000188  -0.20626723
 -0.54663673  0.37336362 -0.33119211 -0.45995065 -0.39305895  0.17315114
  1.80734362  1.34024181  0.32793335  0.54169673  0.40003945  1.75711503
  0.2348163   0.40610619 -0.27040384  0.74169684 -0.37149541 -0.22467869
 -0.89065041 -0.14976776 -0.35416148 -0.35196214 -0.30704198 -1.10837272
 -0.68850645 -0.18754113 -0.25961763  0.44222036  0.71956938  0.31011175
 -0.87380603 -0.25761065 -0.35763975  0.29971468  0.49584234  0.71244868
 -0.12837556 -0.82204431 -0.56970768  0.51936367 -0.25994771 -0.31303677
  0.29745721  0.52000627 -0.43751303  0.81274654 -0.30501018 -0.4858665
 -1.15673013  0.61185731 -0.83667314 -1.04763391 -0.8749282  -0.85027437
 -0.80667006 -1.15082152 -0.5

In [27]:
Data_norm_df

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
AARS,-0.496854,0.288446,0.189747,-0.016454,0.080746,0.282346,0.326246,0.303046,0.387546,0.434746,...,0.846946,0.739186,0.728520,0.661815,0.627668,0.680916,0.700434,0.746670,0.755803,0.789433
ABCB6,-0.658596,-0.142196,-0.075397,-0.383796,-0.199996,-0.074197,0.108804,-0.399196,-0.227496,-0.041146,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ABCC5,-0.080204,0.231996,-0.329354,-0.225204,0.278446,0.034696,0.396596,-0.255704,-0.254154,-0.002004,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ABCF1,0.202535,0.602335,0.403335,0.313134,-0.083265,-0.056365,-0.387216,0.097235,0.540634,-0.262666,...,-0.374807,-0.352552,-0.198751,-0.306206,-0.173180,-0.224540,-0.200641,-0.301348,-0.327189,-0.400115
ABCF3,-0.520919,-0.192819,0.001032,-0.096419,0.210881,-0.731118,0.095381,0.313481,-0.078018,-0.257669,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF395,1.796254,1.773154,0.668354,0.825354,0.350654,0.205954,-0.998346,0.235354,0.230653,0.440854,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ZNF451,-0.244519,0.116732,0.058081,-0.178169,-0.080619,-0.139119,0.064681,-0.083019,-0.272119,-0.164119,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ZNF586,0.097627,0.061027,-0.337573,0.098427,-0.338173,0.017627,-0.197423,-0.078473,-0.300473,-0.147773,...,-0.796426,-0.374528,-0.369905,-0.345740,-0.201382,-0.056259,-0.226123,-0.499067,-0.395016,-0.387449
ZNF589,0.608573,0.106123,-0.014126,-0.003677,-0.123477,-0.243377,0.044623,-0.021477,0.201823,-0.413227,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [28]:
genes = Data_norm_df.index.tolist()
print('Landmark genes list: ', len(genes), genes)

n_genes = len(genes)

Landmark genes list:  978 ['AARS', 'ABCB6', 'ABCC5', 'ABCF1', 'ABCF3', 'ABHD4', 'ABHD6', 'ABL1', 'ACAA1', 'ACAT2', 'ACBD3', 'ACD', 'ACLY', 'ACOT9', 'ADAM10', 'ADAT1', 'ADGRE5', 'ADGRG1', 'ADH5', 'ADI1', 'ADO', 'ADRB2', 'AGL', 'AKAP8', 'AKAP8L', 'AKR7A2', 'AKT1', 'ALAS1', 'ALDH7A1', 'ALDOA', 'ALDOC', 'AMDHD2', 'ANKRD10', 'ANO10', 'ANXA7', 'APBB2', 'APOE', 'APP', 'APPBP2', 'ARFIP2', 'ARHGAP1', 'ARHGEF12', 'ARHGEF2', 'ARID4B', 'ARID5B', 'ARL4C', 'ARNT2', 'ARPP19', 'ASAH1', 'ASCC3', 'ATF1', 'ATF5', 'ATF6', 'ATG3', 'ATMIN', 'ATP11B', 'ATP1B1', 'ATP2C1', 'ATP6V0B', 'ATP6V1D', 'AURKA', 'AURKB', 'AXIN1', 'B4GAT1', 'BACE2', 'BAD', 'BAG3', 'BAMBI', 'BAX', 'BCL2', 'BCL7B', 'BDH1', 'BECN1', 'BHLHE40', 'BID', 'BIRC2', 'BIRC5', 'BLCAP', 'BLMH', 'BLVRA', 'BMP4', 'BNIP3', 'BNIP3L', 'BPHL', 'BRCA1', 'BTK', 'BUB1B', 'BZW2', 'C2CD2', 'C2CD2L', 'C2CD5', 'C5', 'CAB39', 'CALM3', 'CALU', 'CAMSAP2', 'CANT1', 'CAPN1', 'CARMIL1', 'CASC3', 'CASK', 'CASP10', 'CASP2', 'CASP3', 'CASP7', 'CAST', 'CAT', 'CBLB', 'CBR1

## Inhibitor concentrations, IC50, and perturbation matrices

In [29]:
inhib_conc_matrix = np.zeros((n_modules, n_experiments))
ic50_matrix = np.ones((n_modules, n_experiments))
gamma_matrix = np.zeros((n_modules, n_experiments))

In [30]:

for i, module in enumerate(modules):
    drugs_for_module = IC50_df.Drug[IC50_df.Module == module].tolist()
    for drug in drugs_for_module:
        # get IC50 for this drug
        ic50 = IC50_df.IC50[IC50_df.Drug == drug].values
#       gamma = IC50_df.Gamma[IC50_df.Drug == drug].values
        print(drug, ic50)
        assert ic50.size == 1
#       assert gamma.size == 1
        # get experiments with this drug
        exp_with_drug = sig_info_df.index[sig_info_df.pert_drug == drug].tolist()
        print(exp_with_drug) 
        for exp_id in exp_with_drug:
            j = exp_ids.index(exp_id)
            print(j)
            # extract inhibitor concentration
            inhib_conc = sig_info_df.dose_float[sig_info_df.index == exp_id].values
            assert inhib_conc.size == 1
            # insert values in matrices
            inhib_conc_matrix[i, j] = inhib_conc.item()
            ic50_matrix[i, j] = ic50.item()
#           gamma_matrix[i, j] = gamma.item()


flufenamic-acid [3.]
['MOAR008_BC3C_24H:L01', 'MOAR008_BC3C_24H:L02']
76
77
nandrolone [9.]
['MOAR011_BC3C_24H:J10']
103
oxandrolone [190.3]
['MOAR011_BC3C_24H:C01', 'MOAR011_BC3C_24H:C02', 'MOAR011_BC3C_24H:C03']
95
96
97
testosterone-enanthate [200.]
['MOAR011_BC3C_24H:C10', 'MOAR011_BC3C_24H:C11']
98
99
testosterone-propionate [124.]
['MOAR010_BC3C_24H:L19', 'MOAR010_BC3C_24H:L20']
93
94
JNJ-7706621 [0.027]
['ASG002_BC3C_24H:N13', 'ASG002_BC3C_24H:N14', 'ASG002_BC3C_24H:N15']
57
58
59
PHA-793887 [0.18]
['ASG002_BC3C_24H:L01', 'ASG002_BC3C_24H:L02', 'ASG002_BC3C_24H:L03']
46
47
48
roscovitine [2.]
['ASG002_BC3C_24H:E22', 'ASG002_BC3C_24H:E23', 'ASG002_BC3C_24H:E24']
18
19
20
alvocidib [0.12]
['ASG002_BC3C_24H:F05', 'ASG002_BC3C_24H:F06']
24
25
palbociclib [0.045]
['ASG002_BC3C_24H:P16', 'ASG002_BC3C_24H:P18']
72
73
afatinib [0.03]
['ASG002_BC3C_24H:N23', 'ASG002_BC3C_24H:N24']
62
63
erlotinib [0.006]
['ASG002_BC3C_24H:H16', 'ASG002_BC3C_24H:H17', 'ASG002_BC3C_24H:H18']
33
34
35
gefit

In [31]:
inhib_conc_matrix

array([[ 0.,  0.,  0., ...,  0.,  0., 10.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.]])

In [32]:
# transform matrices into pandas dfs for export with row and column names
inhib_conc_df = pd.DataFrame(inhib_conc_matrix, index = modules, columns = exp_ids)
ic50_df = pd.DataFrame(ic50_matrix, index = modules, columns = exp_ids)
# gamma_df = pd.DataFrame(gamma_matrix, index = modules, columns = exp_ids)

# create binary perturbation matrix
pert_df = pd.DataFrame(
    np.where(inhib_conc_matrix != 0, 1, 0),
    index = inhib_conc_df.index,
    columns = inhib_conc_df.columns,
)

In [33]:
display(ic50_df)
# display(gamma_df)
display(inhib_conc_df)
display(pert_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:L20,MOAR011_BC3C_24H:C01,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10
Androgen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,124.0,190.3,190.3,190.3,200.0,200.0,1.00,1.00,1.00,9.0
CDK1_2,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
CDK4_6,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
EGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
Estrogen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
FGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
PI3K,0.00262,0.00262,0.1595,0.1595,0.1595,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
p53,1.00000,1.00000,1.0000,1.0000,1.0000,0.0018,0.0018,9.75,9.75,0.54,...,1.0,1.0,1.0,1.0,1.0,1.0,62.15,62.15,62.15,1.0
TOP2A,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0
Src,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:L20,MOAR011_BC3C_24H:C01,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10
Androgen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,3.33,10.0,3.33,1.11,10.0,3.33,0.0,0.00,0.00,10.0
CDK1_2,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
CDK4_6,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
EGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
Estrogen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
FGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
PI3K,10.0,1.11,10.0,1.11,0.12,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
p53,0.0,0.00,0.0,0.00,0.00,10.0,1.11,1.11,0.08,10.0,...,0.00,0.0,0.00,0.00,0.0,0.00,10.0,3.33,1.11,0.0
TOP2A,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0
Src,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:L20,MOAR011_BC3C_24H:C01,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10
Androgen,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,0,0,0,1
CDK1_2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CDK4_6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
EGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Estrogen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
FGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PI3K,1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
p53,0,0,0,0,0,1,1,1,1,1,...,0,0,0,0,0,0,1,1,1,0
TOP2A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Src,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [34]:
cols = list(tgfb_lfc.columns) + list(smad_lfc.columns)

ic50_df[cols] = 1.0
inhib_conc_df[cols] = 0.0
pert_df[cols] = 0

In [35]:
ic50_df.loc['TGFb',tgfb_lfc.columns] = 200
ic50_df.loc['SMAD3',smad_lfc.columns] = 7

In [36]:
inhib_conc_df.loc['TGFb',tgfb_lfc.columns] = 15
inhib_conc_df.loc['SMAD3',smad_lfc.columns] = 10

In [37]:
pert_df.loc['TGFb',tgfb_lfc.columns] = 1
pert_df.loc['SMAD3',smad_lfc.columns] = 1

In [38]:
display(ic50_df)
# display(gamma_df)
display(inhib_conc_df)
display(pert_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
Androgen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
CDK1_2,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
CDK4_6,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
EGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Estrogen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
FGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
PI3K,0.00262,0.00262,0.1595,0.1595,0.1595,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
p53,1.00000,1.00000,1.0000,1.0000,1.0000,0.0018,0.0018,9.75,9.75,0.54,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
TOP2A,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Src,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
Androgen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CDK1_2,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CDK4_6,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
EGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Estrogen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PI3K,10.0,1.11,10.0,1.11,0.12,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p53,0.0,0.00,0.0,0.00,0.00,10.0,1.11,1.11,0.08,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TOP2A,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Src,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
Androgen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CDK1_2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CDK4_6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
EGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Estrogen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
FGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PI3K,1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
p53,0,0,0,0,0,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
TOP2A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Src,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Global responses for DPD modules

According to our discussion, $R$ for DPD vectors can not be calculated with the same formula as for pathway activities. Instead we are assuming:

\begin{equation}
R_{DPD},j = DPD = STV_{DPD} \cdot Data_j
\end{equation}

In [39]:
# load STV data frame
STVs = pd.read_excel(os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx"), sheet_name = "STV", index_col = 0)
STV_df = pd.DataFrame(np.zeros((len(Data_norm_df.index), 3)), index = Data_norm_df.index, columns = STVs.columns)
STV_df.loc[STVs.index] = STVs

display(STV_df)

,blca_basal_luminal,blca_oncogenesis,blca_survival
AARS,0.0,0.078487,0.000000
ABCB6,0.0,0.027125,-2.890931
ABCC5,0.0,0.002121,0.000000
ABCF1,0.0,-0.028959,-0.496186
ABCF3,0.0,0.036671,0.000000
...,...,...,...
ZNF395,0.0,0.000000,0.000000
ZNF451,0.0,0.013403,0.000000
ZNF586,0.0,0.000000,0.119103
ZNF589,0.0,0.000000,0.000000


In [40]:
# create empty DPD data frame
DPD_df = pd.DataFrame(
    np.zeros((len(Data_norm_df.columns), len(STV_df.columns))),
    index = Data_norm_df.columns,
    columns = STV_df.columns,
)

# populate
for exp_id in DPD_df.index:
    for state in STV_df.columns:
        DPD_df.loc[exp_id, state] = np.dot(Data_norm_df.T.loc[exp_id], STV_df.loc[:, state])

display(DPD_df)

,blca_basal_luminal,blca_oncogenesis,blca_survival
ASG002_BC3C_24H:A10,-56.563525,-0.610650,-17.377163
ASG002_BC3C_24H:A11,-51.696023,-0.936049,-5.764659
ASG002_BC3C_24H:A19,-7.483112,-0.048496,9.872480
ASG002_BC3C_24H:A20,3.688500,-0.647673,-0.584775
ASG002_BC3C_24H:A21,-12.723954,0.031650,-6.200738
...,...,...,...
SMAD3in_V16,-46.656824,-0.604149,-12.661158
SMAD3in_V17,-47.118911,-0.604757,-17.312730
SMAD3in_V18,-53.112170,-0.868735,-19.128066
SMAD3in_V19,-52.279880,-0.735904,-16.796479


In [41]:
# transform to R global
R_global_DPD_df = DPD_df.T
display(R_global_DPD_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
blca_basal_luminal,-56.563525,-51.696023,-7.483112,3.688500,-12.723954,-14.104169,4.258746,-26.565452,-5.792472,-15.570078,...,-67.688514,-47.014115,-45.027390,-45.999619,-44.159687,-46.656824,-47.118911,-53.112170,-52.279880,-60.038526
blca_oncogenesis,-0.610650,-0.936049,-0.048496,-0.647673,0.031650,0.185978,0.911126,-0.079387,-0.459741,-0.521443,...,-1.136658,-0.816257,-0.719123,-0.512626,-0.501798,-0.604149,-0.604757,-0.868735,-0.735904,-1.067935
blca_survival,-17.377163,-5.764659,9.872480,-0.584775,-6.200738,1.836483,-3.060844,-3.581400,1.041249,-6.062045,...,-26.074401,-17.264842,-16.611898,-12.734232,-12.923143,-12.661158,-17.312730,-19.128066,-16.796479,-22.067899


## Save outputs

In [42]:
# save metadata as pickle
all_metadata = {
    "modules": modules,
    "n_modules": n_modules,
    "drugs": drugs,
    "n_drugs": n_drugs,
    "exp_ids": exp_ids,
    "n_experiments": n_experiments,
    "genes": genes,
    "n_genes": n_genes,
}

print(all_metadata)

with open(os.path.join(out_dir, "metadata.pickle"), "wb") as f:
    pickle.dump(all_metadata, f, protocol = pickle.HIGHEST_PROTOCOL)

{'modules': ['Androgen', 'CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'TGFb', 'SMAD3'], 'n_modules': 12, 'drugs': ['estradiol-cypionate', 'nutlin-3', 'sorafenib', 'NVP-BEZ235', 'AZD-8055', 'idarubicin', 'PI-103', 'Agent1', 'PHA-793887', 'KU-0063794', 'erlotinib', 'GDC-0349', 'Agent2', 'alvocidib', 'SAR405838', 'JNJ-7706621', 'testosterone-propionate', 'HLI-373', 'AMG-232', 'afatinib', 'flufenamic-acid', 'roscovitine', 'AS-605240', 'vandetanib', 'lapatinib', 'dasatinib', 'epirubicin', 'serdemetan', 'RITA', 'nandrolone', 'ponatinib', 'testosterone-enanthate', 'LY-294002', 'daunorubicin', 'palbociclib', 'gefitinib', 'raloxifene', 'taselisib', 'masitinib', 'dienestrol', 'mitoxantrone', 'oxandrolone'], 'n_drugs': 42, 'exp_ids': ['ASG002_BC3C_24H:A10', 'ASG002_BC3C_24H:A11', 'ASG002_BC3C_24H:A19', 'ASG002_BC3C_24H:A20', 'ASG002_BC3C_24H:A21', 'ASG002_BC3C_24H:B10', 'ASG002_BC3C_24H:B11', 'ASG002_BC3C_24H:B14', 'ASG002_BC3C_24H:B15', 'ASG002_BC3C_24H:C13', 'A

In [43]:
# save doses and perturbation matrix
inhib_conc_df.to_csv(os.path.join(out_dir, "inhib_conc_annotated.csv"))
ic50_df.to_csv(os.path.join(out_dir, "ic50_annotated.csv"))
# gamma_df.to_csv(os.path.join(out_dir, "gamma_annotated.csv"))
pert_df.to_csv(os.path.join(out_dir, "pert_annotated.csv"))

In [44]:
# save log fold change L1000 data
Data_norm_df.to_csv(os.path.join(out_dir, "L1000_Data_norm_data.csv"))

In [45]:
# save R_global for DPDs
R_global_DPD_df.to_csv(os.path.join(out_dir, "R_global_DPDonly_annotated.csv"))